# Extreme Weather Anomaly Tracking and Downscaling (Kaggle GPU Pipeline)

This notebook trains and evaluates the complete two-stage hybrid pipeline:
1. **Stage 1 (Tracker):** Spherical Icosahedral Multi-Mesh Graph Neural Network for detecting and tracking extreme anomalies across lead times (Day 3 to Day 10).
2. **Stage 2 (Downscaler):** CorrDiff-style conditional residual diffusion model downscaling from 12 km to 5 km while **preserving extreme disaster amplitudes** using physics-informed loss constraints.
3. **Alert Engine & Verification:** Hyper-local 5 km radius alerts and full probabilistic evaluation.

### Step 1: Environment Verification (GPU & CUDA)

In [ ]:
import torch
import os
import sys

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0)/(1024**2):.1f} MB")
    device = "cuda"
else:
    print("Running on CPU (use GPU accelerator in Kaggle settings for accelerated training)")
    device = "cpu"

### Step 2: Install Core Pinned Dependencies and Run Smoke Tests

In [ ]:
# Clone or navigate to the repository directory
!git clone https://github.com/aditrisingh/weather-anomaly.git /kaggle/working/weather-anomaly 2>/dev/null || true
%cd /kaggle/working/weather-anomaly 2>/dev/null || cd .

!pip install -q pyyaml scipy pytest pydantic fastapi httpx uvicorn
!pytest tests/ -v

### Step 3: Train Stage 1 Spherical GNN Tracker

In [ ]:
!python scripts/train_tracker.py --config configs/tracker.yaml --epochs 5 --device {device}

### Step 4: Train Stage 2 CorrDiff Downscaler with Physics Constraints

In [ ]:
!python scripts/train_downscaler.py --config configs/downscaler.yaml --epochs 5 --device {device}

### Step 5: Run Full End-to-End Cycle Inference & Alert Generation

In [ ]:
!python scripts/infer.py --config configs/infer.yaml

### Step 6: Inspect Output Alerts & 5 km Core Impact Geometry

In [ ]:
import json

output_file = "data/outputs/cycles/cycle_2026-10-02T00Z.json"
with open(output_file, "r") as f:
    cycle_results = json.load(f)

print(f"Detected Tracks: {len(cycle_results['tracks'])}")
print(f"Generated Alerts: {cycle_results['count_alerts']}")
for alert in cycle_results['alerts']:
    print(f"[*] Alert {alert['alert_id']}: Category {alert['category'].upper()}")
    print(f"    Core Location: Lat {alert['core']['lat']}, Lon {alert['core']['lon']}")
    print(f"    Peak Intensity: {alert['peak_value']} mm/day | Exceedance Prob: {alert['exceedance_prob']:.1%}")
    print(f"    Valid Window: {alert['valid_from']} to {alert['valid_to']}")